In [1]:
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
from pathlib import Path
import random
import numpy as np
import torch
import pandas as pd 
import time

from models_transformer import SingleOutTransformerNet
from shap_helpers import (compute_shap_3out, 
                          _ensure_2d_shap, 
                          mean_abs_shap, 
                          topk_table, 
                          global_table,
                          save_fig, 
                          cosine_spearman)

In [2]:
OUTDIR = Path("./Results")
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


In [3]:
mpl.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

TOPK = 10  # MOD: Top 10 as requested
N_BG = 256
SEED_SHAP = SEED

SHAP_DIR = OUTDIR / "shap_values"
FIG_DIR = SHAP_DIR / "figs"
TAB_DIR = SHAP_DIR / "tables"
VAL_DIR = SHAP_DIR / "values"
SHAP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
VAL_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
test_data = np.load("data_processed/test_data_scaled.npz")
X_test = test_data["x"]

In [5]:
rng = np.random.default_rng(SEED_SHAP)
n_test = X_test.shape[0]
idx_bg = rng.choice(n_test, size=min(N_BG, n_test), replace=False)
X_bg = X_test[idx_bg].astype(np.float32)
X_ex = X_test.astype(np.float32)  # explain all test

In [6]:
import pickle

with open("data_processed/dataframes/input_cols_pruned.pkl", "rb") as f:
    INPUT_COLS_PRUNED = pickle.load(f)

feature_names = list(INPUT_COLS_PRUNED)

In [7]:
bg_t = torch.from_numpy(X_bg).to(DEVICE)
ex_t = torch.from_numpy(X_ex).to(DEVICE)

In [8]:
# -----------------------------
# SHAP for TR single-output 
# -----------------------------
# print("\n[SHAP] Computing for TR single-output models...")

class SinglePredictWrapper(torch.nn.Module):
    def __init__(self, model_single):
        super().__init__()
        self.model = model_single
    def forward(self, x):
        return self.model(x)

IN_DIM = X_test.shape[1]

EMB_DIM = 64
NHEAD = 4
NUM_LAYERS = [5, 7, 9, 11, 13, 15, 17]
FF_DIM = 128
DROPOUT = 0.1

In [9]:
for layer in NUM_LAYERS:

    print(f"=== {layer} layers: starting SHAP computation ===")
    
    start = time.time()
    model = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                    num_layers=layer, ff_dim=FF_DIM, 
                                    dropout=DROPOUT).to(DEVICE)

    sd = torch.load(f"trained_models/TR_model_{layer}layers.pt")
    model.load_state_dict(sd)
    model.eval()

    wrapped = SinglePredictWrapper(model).to(DEVICE)
    expl = shap.GradientExplainer(wrapped, bg_t)
    sv = expl.shap_values(ex_t)
    method_s = "GradientExplainer"

    if isinstance(sv, list):
        sv = sv[0]
    
    sv2 = _ensure_2d_shap(sv, tag="age")

    ma = mean_abs_shap(sv2, tag="age")

    df_top = topk_table(ma, feature_names, TOPK, tabdir=TAB_DIR / f"{layer}layers")
    _ = global_table(ma, feature_names, tabdir=TAB_DIR / f"{layer}layers")

    top_feats = df_top["feature"].tolist()
    top_idx = [feature_names.index(f) for f in top_feats]

    shap.summary_plot(sv2[:, top_idx], 
                      features=X_ex[:, top_idx], 
                      feature_names=top_feats, 
                      cmap="viridis", 
                      show=False)

    save_fig(FIG_DIR / f"{layer}layers/beeswarm_top{TOPK}_{layer}layers")

    np.savez_compressed(VAL_DIR / f"{layer}layers/shap_values_{layer}layers.npz",
                        X_ex=X_ex, X_bg=X_bg,
                        shap_vals=sv2,
                        feature_names=np.array(feature_names, dtype=object),
                        method=np.array([method_s], dtype=object)
                       )

    end = time.time()
    
    print(f"=== {layer} layers: SHAP computation completed ===")
    print(f"total time: {round(end-start, 2)} sec.")


=== 5 layers: starting SHAP computation ===


/tmp/ipykernel_152181/694789056.py:32: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv2[:, top_idx],


=== 5 layers: SHAP computation completed ===
total time: 251.69 sec.
=== 7 layers: starting SHAP computation ===


/tmp/ipykernel_152181/694789056.py:32: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv2[:, top_idx],


=== 7 layers: SHAP computation completed ===
total time: 369.6 sec.
=== 9 layers: starting SHAP computation ===


...

=== 13 layers: SHAP computation completed ===
total time: 533.97 sec.
=== 15 layers: starting SHAP computation ===


/tmp/ipykernel_152181/694789056.py:32: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv2[:, top_idx],


=== 15 layers: SHAP computation completed ===
total time: 591.67 sec.
=== 17 layers: starting SHAP computation ===


/tmp/ipykernel_152181/694789056.py:32: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(sv2[:, top_idx],


=== 17 layers: SHAP computation completed ===
total time: 772.26 sec.
